# TSP_DRL_PtrNet — Reproduction from a clean start

This notebook rebuilds the PFE experiment around the **upstream implementation**:
[Rintarooo/TSP_DRL_PtrNet](https://github.com/Rintarooo/TSP_DRL_PtrNet), implementing Bello et al. (2016), *Neural Combinatorial Optimization with Reinforcement Learning*.

## Experiment contract

1. Clone the public repository and record the exact Git commit and Colab environment.
2. Preserve the upstream training script and apply a small, documented patch: seed the run, disable the upstream early-stop heuristic (it compares cumulative training cost rather than a held-out validation metric), and save actor-only evaluation snapshots periodically.
3. Run a small GPU smoke test, then train the 20-city Actor–Critic model from random initialization with the repository defaults (batch size 512, 13,000 optimizer steps by default).
4. Create and persist fixed test instances for 20, 50, and 100 cities.
5. Benchmark nearest neighbor, nearest neighbor + 2-opt, and ACO on the **same** instances. Evaluate the trained checkpoints using greedy, one-sample stochastic, and best-of-512 sampling inference.
6. Save raw per-instance tour lengths and runtimes before summarizing results. Calculate relative gaps against one common best-observed reference for each test instance. These are **not optimality gaps** unless an exact optimum is independently known.
7. Only after the run is complete should its verified outputs be used to revise Chapter IV of the thesis.

### Important methodological notes

- This notebook uses the repository's `actor.py`, `critic.py`, `env.py`, `data.py`, and core `train.py` algorithm. The patch is recorded as a diff and must be disclosed in the thesis.
- A checkpoint is an **actor evaluation snapshot**, not a full resumable training state. Keep the Colab session running through the full run. If training is interrupted, the snapshots and logs already written to Drive are retained, but continuing training from one of them would require a separate controlled resume procedure.
- The repository was written for an older PyTorch environment. This notebook records the current environment rather than pretending it is identical to the author's original setup.
- The current benchmark's ACO implementation and parameters are explicitly included below. They are a reproducible baseline, not a claim to represent the state of the art.


## Step 0 — Configure Colab

In Google Colab, select **Runtime → Change runtime type → GPU** before running the notebook. A T4 is suitable as a starting point; the actual assigned GPU will be recorded below.

Run cells from top to bottom. Training outputs are written to Google Drive so that completed artifacts survive a runtime reset.

In [ ]:
from pathlib import Path
from datetime import datetime
import os, sys, json, shutil, subprocess, platform, hashlib, re, difflib, time
import numpy as np

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Open this notebook in Google Colab, then select a GPU runtime.") from exc

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/PFE_TSP_DRL')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

# Stable run ID: change v1 to v2 for a genuinely new independent training run.
RUN_ID = 'clean_restart_seed20261013_v1'
RUN_ROOT = PROJECT_ROOT / 'runs' / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)

TRAIN_SEED = 20261013
TEST_SEED = 4202026
N_TEST_INSTANCES = 100
TRAIN_CITY_COUNT = 20
TRAIN_BATCH_SIZE = 512
TRAIN_STEPS = 13000  # upstream README example; set lower only for a pilot run
CHECKPOINT_EVERY = 1000

print('Project root:', PROJECT_ROOT)
print('Run root:', RUN_ROOT)
print('Run ID:', RUN_ID)
print('Training seed:', TRAIN_SEED)
print('Test seed:', TEST_SEED)
print('Test instances per size:', N_TEST_INSTANCES)

In [ ]:
import torch
print('Python:', sys.version.replace('\n', ' '))
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA runtime reported by PyTorch:', torch.version.cuda)
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
    subprocess.run(['nvidia-smi'], check=False)
else:
    raise RuntimeError('GPU is not available. In Colab, select Runtime → Change runtime type → GPU, then rerun this cell.')

## Step 1 — Clone and fingerprint the upstream repository

We do not train from an old modified local copy. The notebook clones the public upstream repository into the ephemeral Colab filesystem, records its commit, and stores an experiment manifest on Drive.

In [ ]:
REPO_URL = 'https://github.com/Rintarooo/TSP_DRL_PtrNet.git'
REPO_DIR = Path('/content/TSP_DRL_PtrNet')
if not (REPO_DIR / '.git').exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Using already-cloned repository in this Colab runtime.')

COMMIT_SHA = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
COMMIT_SHORT = COMMIT_SHA[:12]
print('Repository:', REPO_URL)
print('Commit:', COMMIT_SHA)
print('Top-level files:', ', '.join(sorted(p.name for p in REPO_DIR.iterdir())[:40]))

for required in ['actor.py', 'critic.py', 'env.py', 'data.py', 'config.py', 'train.py', 'search.py', 'test.py']:
    if not (REPO_DIR / required).is_file():
        raise FileNotFoundError(f'Expected upstream file is missing: {required}')

manifest = {
    'project': 'TSP_DRL_PtrNet clean restart',
    'repository_url': REPO_URL,
    'commit_sha': COMMIT_SHA,
    'run_id': RUN_ID,
    'train_seed': TRAIN_SEED,
    'test_seed': TEST_SEED,
    'n_test_instances_per_size': N_TEST_INSTANCES,
    'city_sizes': [20, 50, 100],
    'train_city_count': TRAIN_CITY_COUNT,
    'train_batch_size': TRAIN_BATCH_SIZE,
    'train_steps_target': TRAIN_STEPS,
    'checkpoint_every': CHECKPOINT_EVERY,
    'created_at_local': datetime.now().isoformat(timespec='seconds'),
}
(RUN_ROOT / 'manifest_pre.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Manifest written to Drive.')

## Step 2 — Install lightweight dependencies and save the environment

Colab supplies PyTorch. We install only the small utility packages needed by the original repository, without pinning an obsolete PyTorch version that may conflict with Colab's current CUDA runtime.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tqdm', 'matplotlib', 'numpy', 'pandas'], check=True)
print('Dependency versions:')
subprocess.run([sys.executable, '-m', 'pip', 'show', 'torch', 'numpy', 'tqdm', 'matplotlib'], check=False)
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(RUN_ROOT / 'pip-freeze.txt').write_text(freeze, encoding='utf-8')
(RUN_ROOT / 'git-commit.txt').write_text(COMMIT_SHA + '\n', encoding='utf-8')
print('Environment record saved to:', RUN_ROOT)

## Step 3 — Apply and record the minimal training patch

Two issues make an unmodified run less suitable for this reproducibility study:

1. The upstream early-stop condition compares the **cumulative mean training tour length** against the best cumulative mean seen so far. It is not a validation-based early stopper, so it can terminate the run before the configured step budget.
2. The upstream trainer saves the actor at the end, but not at regular intervals. Periodic actor-only snapshots are useful for analysis if the runtime fails; they do not provide optimizer/critic state for resuming training.

We retain the Actor, Critic, reward/cost, policy-gradient loss, optimizers, learning-rate schedule, data generator, and model hyperparameters. The exact patch is saved to `training_patch.diff` and must be described in the thesis.

In [ ]:
# Always patch the pristine file from the checked-out Git commit, not a possibly
# partially edited train.py. This also makes rerunning this cell safe.
train_path = REPO_DIR / 'train.py'
upstream_train = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'show', 'HEAD:train.py'], text=True
)
(RUN_ROOT / 'train.py.upstream').write_text(upstream_train, encoding='utf-8')
patched_train = upstream_train

# Add os import if absent. Work line-by-line rather than matching one exact block.
lines = patched_train.splitlines(keepends=True)
if not any(line.strip() == 'import os' for line in lines):
    import_torch_idx = next((i for i, line in enumerate(lines) if line.strip() == 'import torch'), None)
    if import_torch_idx is None:
        raise RuntimeError('Could not find "import torch" in upstream train.py.')
    lines.insert(import_torch_idx, 'import os\n')
patched_train = ''.join(lines)

# Disable only the upstream cumulative-training-cost early-stop block. The repository
# does not use a held-out validation set for this condition, so our planned run uses
# the configured fixed number of steps. Locate the block by its code, not indentation.
lines = patched_train.splitlines(keepends=True)
stop_matches = [i for i, line in enumerate(lines)
                if re.search(r'^\s*if\s*\(\s*ave_L\s*/\s*\(\s*i\s*\+\s*1\s*\)\s*<\s*min_L\s*\)\s*:', line)]
if len(stop_matches) != 1:
    preview = '\n'.join(f'{i+1}: {line.rstrip()}' for i, line in enumerate(lines)
                        if 'ave_L' in line or 'early stop' in line or 't1 = time()' in line)
    raise RuntimeError(f'Expected one cumulative-mean early-stop block; found {len(stop_matches)}. Relevant lines:\n{preview}')
stop_start = stop_matches[0]
indent = lines[stop_start][:len(lines[stop_start]) - len(lines[stop_start].lstrip(' \t'))]
stop_end = None
for j in range(stop_start + 1, len(lines)):
    candidate = lines[j]
    candidate_indent = candidate[:len(candidate) - len(candidate.lstrip(' \t'))]
    if candidate.strip() == 't1 = time()' and candidate_indent == indent:
        stop_end = j
        break
if stop_end is None:
    raise RuntimeError('Found the early-stop condition but could not locate its matching t1 = time() boundary.')
newline = '\r\n' if (lines[stop_start].endswith('\r\n')) else '\n'
replacement = [
    indent + '# Fixed-step experiment patch: disable cumulative-mean early stopping.\n',
    indent + 'current_mean_cost = ave_L / (i + 1)\n',
    indent + 'if current_mean_cost < min_L:\n',
    indent + '\tmin_L = current_mean_cost\n',
]
lines = lines[:stop_start] + replacement + lines[stop_end:]
patched_train = ''.join(lines)

# Set Python, NumPy and PyTorch RNG seeds before constructing the training environment.
lines = patched_train.splitlines(keepends=True)
main_matches = [i for i, line in enumerate(lines)
                if line.strip() == 'cfg = load_pkl(pkl_parser().path)']
if len(main_matches) != 1:
    raise RuntimeError(f'Expected one train.py main config-load line; found {len(main_matches)}.')
main_idx = main_matches[0]
indent = lines[main_idx][:len(lines[main_idx]) - len(lines[main_idx].lstrip(' \t'))]
seed_lines = [
    indent + 'import random\n',
    indent + 'import numpy as np\n',
    indent + 'seed = int(getattr(cfg, "seed", 1))\n',
    indent + 'random.seed(seed)\n',
    indent + 'np.random.seed(seed)\n',
    indent + 'torch.manual_seed(seed)\n',
    indent + 'if torch.cuda.is_available():\n',
    indent + '\ttorch.cuda.manual_seed_all(seed)\n',
    indent + 'torch.backends.cudnn.deterministic = True\n',
    indent + 'torch.backends.cudnn.benchmark = False\n',
]
lines[main_idx + 1:main_idx + 1] = seed_lines
patched_train = ''.join(lines)

# Periodically save actor-only snapshots after the actor optimizer update. The final
# model is saved by the original end-of-training block below.
lines = patched_train.splitlines(keepends=True)
actor_loss_matches = [i for i, line in enumerate(lines)
                      if re.search(r'^\s*ave_act_loss\s*\+=\s*act_loss\.item\(\)', line)]
if len(actor_loss_matches) != 1:
    preview = '\n'.join(f'{i+1}: {line.rstrip()}' for i, line in enumerate(lines)
                        if 'act_optim.step' in line or 'act_lr_scheduler.step' in line or 'ave_act_loss' in line)
    raise RuntimeError(f'Expected one actor-loss accumulator line; found {len(actor_loss_matches)}. Relevant lines:\n{preview}')
insert_at = actor_loss_matches[0]
indent = lines[insert_at][:len(lines[insert_at]) - len(lines[insert_at].lstrip(' \t'))]
periodic_lines = [
    indent + 'completed_steps = i + 1\n',
    indent + "checkpoint_every = int(os.environ.get('TSP_DRL_CHECKPOINT_EVERY', '1000'))\n",
    indent + 'if cfg.issaver and checkpoint_every > 0 and completed_steps % checkpoint_every == 0 and completed_steps < cfg.steps:\n',
    indent + '\tcheckpoint_name = f"{cfg.task}_{date}_step{completed_steps}_act.pt"\n',
    indent + '\tcheckpoint_path = os.path.join(cfg.model_dir, checkpoint_name)\n',
    indent + '\ttorch.save(act_model.state_dict(), checkpoint_path)\n',
    indent + '\tprint(f"saved periodic actor checkpoint: {checkpoint_path}")\n',
]
lines[insert_at:insert_at] = periodic_lines
patched_train = ''.join(lines)

# Keep the upstream final-save statement unchanged. Repository versions can use
# different filename formatting, and renaming it is not needed for periodic snapshots.
# The upstream final save still runs at the end of training. Periodic snapshots above
# provide intermediate actor weights only (not a resumable optimizer/critic state).

# Save the applied patch and compile before training.
train_path.write_text(patched_train, encoding='utf-8')
patch_text = ''.join(difflib.unified_diff(
    upstream_train.splitlines(keepends=True), patched_train.splitlines(keepends=True),
    fromfile='a/train.py (upstream)', tofile='b/train.py (experiment patch)'))
(RUN_ROOT / 'training_patch.diff').write_text(patch_text, encoding='utf-8')
print(patch_text)
print('\nPatched train.py syntax check...')
import py_compile
py_compile.compile(str(train_path), doraise=True)
print('PASS: train.py compiles.')
print('PASS: patch based on pristine Git version; original upstream final-save statement preserved.')


## Step 4 — Smoke test on the GPU

Run a deliberately small training job first. This verifies the environment, data generation, actor–critic forward/backward passes, logging, and checkpoint writing before using the full training budget. The smoke-test weights are kept separately and are **not** used as the final model.

In [ ]:
def prepare_training_config(run_dir, batch_size, steps, seed, log_step=100):
    run_dir = Path(run_dir)
    csv_dir, pt_dir, pkl_dir = run_dir / 'Csv', run_dir / 'Pt', run_dir / 'Pkl'
    for d in [csv_dir, pt_dir, pkl_dir]:
        d.mkdir(parents=True, exist_ok=True)
    pkl_path = pkl_dir / f'train{TRAIN_CITY_COUNT}.pkl'
    if pkl_path.exists():
        print('Using existing config:', pkl_path)
        return pkl_path
    cmd = [
        sys.executable, 'config.py', '-m', 'train',
        '-b', str(batch_size), '-t', str(TRAIN_CITY_COUNT), '-s', str(steps),
        '-e', '128', '-hi', '128', '-c', '10', '-st', '1.0',
        '-o', 'Adam', '-ng', '1', '-np', '3', '-dt', 'sampling',
        '--lr', '0.001', '--seed', str(seed), '-ls', str(log_step),
        '-ld', str(csv_dir) + os.sep,
        '-md', str(pt_dir) + os.sep,
        '-pd', str(pkl_dir) + os.sep,
    ]
    result = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True, check=True)
    print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
    config_json = {
        'mode': 'train', 'city_t': TRAIN_CITY_COUNT,
        'batch': batch_size, 'steps': steps, 'seed': seed,
        'embed': 128, 'hidden': 128, 'clip_logits': 10,
        'softmax_T': 1.0, 'optim': 'Adam', 'n_glimpse': 1,
        'n_process': 3, 'decode_type': 'sampling', 'lr': 0.001,
        'lr_decay': 0.96, 'lr_decay_step': 5000,
        'fixed_step_budget': True, 'checkpoint_every': CHECKPOINT_EVERY,
    }
    (run_dir / 'config.json').write_text(json.dumps(config_json, indent=2), encoding='utf-8')
    if not pkl_path.exists():
        raise FileNotFoundError(f'config.py did not create expected config: {pkl_path}')
    return pkl_path

def run_training(run_dir, pkl_path, checkpoint_every=1000):
    run_dir = Path(run_dir)
    pt_dir = run_dir / 'Pt'
    # Prevent silently overwriting/duplicating an existing run. Change RUN_ID for a new run.
    if any(pt_dir.glob('*_step*_act.pt')):
        raise RuntimeError(f'Actor checkpoints already exist in {pt_dir}. Do not rerun training over them; use a new RUN_ID for a fresh run.')
    env = os.environ.copy()
    env['TSP_DRL_CHECKPOINT_EVERY'] = str(checkpoint_every)
    log_path = run_dir / 'training_stdout.log'
    command = [sys.executable, 'train.py', '-p', str(pkl_path)]
    print('Command:', ' '.join(command))
    with log_path.open('w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        code = proc.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, command)
    ckpts = sorted(pt_dir.glob('*_step*_act.pt'))
    print(f'\nTraining process finished. Actor checkpoint files: {len(ckpts)}')
    for p in ckpts:
        print(f'  {p.name} ({p.stat().st_size / (1024**2):.2f} MiB)')
    return ckpts

SMOKE_DIR = RUN_ROOT / 'smoke_test'
smoke_pkl = prepare_training_config(SMOKE_DIR, batch_size=32, steps=30, seed=TRAIN_SEED, log_step=10)
smoke_ckpts = run_training(SMOKE_DIR, smoke_pkl, checkpoint_every=10)
print('Smoke-test artifacts:', SMOKE_DIR)

## Step 5 — Train the main model from random initialization

The default is a 20-city Actor–Critic run with batch size 512 and **13,000 configured steps**, matching the example in the upstream README. The dataset generator creates `batch × steps` instances before training; for these defaults, the coordinate tensor alone is about 1.06 GiB in float32 on the selected device. Do not increase the batch or step count until this run is stable.

The model is trained from scratch: no pretrained actor weights are loaded. Periodic snapshots are written to Drive at the configured interval and at the final step.

In [ ]:
FULL_RUN_DIR = RUN_ROOT / 'full_train_n20'
full_pkl = prepare_training_config(FULL_RUN_DIR, batch_size=TRAIN_BATCH_SIZE,
                                  steps=TRAIN_STEPS, seed=TRAIN_SEED, log_step=100)
expected_dataset_gib = TRAIN_BATCH_SIZE * TRAIN_STEPS * TRAIN_CITY_COUNT * 2 * 4 / (1024**3)
print(f'Approx. training-coordinate tensor size (float32): {expected_dataset_gib:.2f} GiB')
full_checkpoints = run_training(FULL_RUN_DIR, full_pkl, checkpoint_every=CHECKPOINT_EVERY)

training_manifest = {
    **manifest,
    'commit_sha': COMMIT_SHA,
    'full_run_dir': str(FULL_RUN_DIR),
    'training_steps_configured': TRAIN_STEPS,
    'training_batch_size': TRAIN_BATCH_SIZE,
    'training_coordinate_tensor_estimated_gib': expected_dataset_gib,
    'actor_checkpoints': [{'path': str(p), 'bytes': p.stat().st_size,
                           'sha256': hashlib.sha256(p.read_bytes()).hexdigest()}
                          for p in sorted((FULL_RUN_DIR / 'Pt').glob('*_step*_act.pt'))],
}
(RUN_ROOT / 'manifest_training.json').write_text(json.dumps(training_manifest, indent=2), encoding='utf-8')
print('Saved training manifest:', RUN_ROOT / 'manifest_training.json')

## Step 6 — Inspect the training curves and saved checkpoints

We do not call the model “converged” merely because a loss curve flattens. This cell plots the repository's recorded average actor loss, critic loss, and average sampled tour length, and lists all saved actor snapshots.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

train_logs = list((FULL_RUN_DIR / 'Csv').glob('*_train.csv'))
if not train_logs:
    raise FileNotFoundError(f'No training CSV found in {FULL_RUN_DIR / "Csv"}')
log_file = max(train_logs, key=lambda p: p.stat().st_mtime)
train_df = pd.read_csv(log_file)
print('Training log:', log_file)
display(train_df.tail(10))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(train_df['step'], train_df['average distance'])
axes[0].set(title='Cumulative mean sampled tour length', xlabel='Training step', ylabel='Tour length')
axes[0].grid(True, alpha=.3)
axes[1].plot(train_df['step'], train_df['actic loss'], label='Actor loss')
if 'critic loss' in train_df.columns:
    axes[1].plot(train_df['step'], train_df['critic loss'], label='Critic loss')
axes[1].set(title='Recorded training losses', xlabel='Training step', ylabel='Loss')
axes[1].legend(); axes[1].grid(True, alpha=.3)
plt.tight_layout()
curve_path = RUN_ROOT / 'training_curves.png'
fig.savefig(curve_path, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', curve_path)

checkpoint_files = sorted((FULL_RUN_DIR / 'Pt').glob('*_step*_act.pt'))
for p in checkpoint_files:
    print(p.name, f'{p.stat().st_size / (1024**2):.2f} MiB')

## Step 7 — Generate fixed test instances

One saved set of 100 instances is used for every method at each problem size. The coordinates are independent uniform points in `[0, 1]²`, matching the input distribution used by the repository's generator. The test seed and instances are persisted so later reruns use exactly the same inputs.

In [ ]:
BENCH_DIR = RUN_ROOT / 'benchmark'
INSTANCE_DIR = BENCH_DIR / 'instances'
INSTANCE_DIR.mkdir(parents=True, exist_ok=True)

TEST_COORDS = {}
for n in [20, 50, 100]:
    npz_path = INSTANCE_DIR / f'test_n{n}_seed{TEST_SEED}_count{N_TEST_INSTANCES}.npz'
    if npz_path.exists():
        archive = np.load(npz_path)
        coords = archive['coords']
        if coords.shape != (N_TEST_INSTANCES, n, 2):
            raise ValueError(f'Existing test file has unexpected shape: {coords.shape}; choose a new TEST_SEED or count.')
        print('Loaded existing fixed test set:', npz_path)
    else:
        rng = np.random.default_rng(TEST_SEED + n)
        coords = rng.random((N_TEST_INSTANCES, n, 2), dtype=np.float64).astype(np.float32)
        np.savez_compressed(npz_path, coords=coords, n=np.int64(n), seed=np.int64(TEST_SEED), count=np.int64(N_TEST_INSTANCES))
        print('Created fixed test set:', npz_path)
    TEST_COORDS[n] = coords

instance_manifest = {
    'test_seed': TEST_SEED,
    'instances_per_size': N_TEST_INSTANCES,
    'distribution': 'independent uniform coordinates in [0,1]^2',
    'city_sizes': {str(n): {'shape': list(TEST_COORDS[n].shape),
                            'file': str(INSTANCE_DIR / f'test_n{n}_seed{TEST_SEED}_count{N_TEST_INSTANCES}.npz')}
                   for n in TEST_COORDS},
}
(BENCH_DIR / 'test_instances_manifest.json').write_text(json.dumps(instance_manifest, indent=2), encoding='utf-8')
print('All test sets use identical instances across all methods.')

## Step 8 — Define the classical baselines

All costs use the same Euclidean closed-cycle tour-length calculation. The NN start city is fixed at index 0. `NN + 2-opt` starts from that same NN tour and applies first-improvement 2-opt until no improving edge swap is found or the pass cap is reached. ACO uses the fixed settings below for every size and instance; no parameter search is performed on the test set.

In [ ]:
def tour_length_np(coords, tour):
    coords = np.asarray(coords, dtype=np.float64)
    tour = np.asarray(tour, dtype=np.int64)
    ordered = coords[tour]
    return float(np.linalg.norm(ordered[1:] - ordered[:-1], axis=1).sum()
                 + np.linalg.norm(ordered[0] - ordered[-1]))

def distance_matrix(coords):
    delta = coords[:, None, :] - coords[None, :, :]
    return np.sqrt(np.sum(delta * delta, axis=-1))

def nearest_neighbor(coords, start=0):
    coords = np.asarray(coords, dtype=np.float64)
    n = len(coords)
    d = distance_matrix(coords)
    tour = [int(start)]
    unvisited = set(range(n))
    unvisited.remove(int(start))
    while unvisited:
        current = tour[-1]
        nxt = min(unvisited, key=lambda j: (d[current, j], j))
        tour.append(nxt)
        unvisited.remove(nxt)
    return np.asarray(tour, dtype=np.int64)

def two_opt(coords, initial_tour, max_passes=100):
    coords = np.asarray(coords, dtype=np.float64)
    d = distance_matrix(coords)
    tour = np.asarray(initial_tour, dtype=np.int64).copy()
    n = len(tour)
    for _ in range(max_passes):
        improved = False
        for i in range(n - 1):
            for j in range(i + 2, n):
                if i == 0 and j == n - 1:
                    continue  # the two edges would be adjacent on the cycle
                a, b = tour[i], tour[(i + 1) % n]
                c, e = tour[j], tour[(j + 1) % n]
                old_cost = d[a, b] + d[c, e]
                new_cost = d[a, c] + d[b, e]
                if old_cost > new_cost + 1e-12:
                    tour[i + 1:j + 1] = tour[i + 1:j + 1][::-1]
                    improved = True
                    break
            if improved:
                break
        if not improved:
            break
    return tour

ACO_ANTS = 15
ACO_ITERATIONS = 30
ACO_ALPHA = 1.0
ACO_BETA = 3.0
ACO_RHO = 0.1
ACO_Q = 1.0

def ant_colony_tsp(coords, seed, n_ants=ACO_ANTS, n_iter=ACO_ITERATIONS,
                   alpha=ACO_ALPHA, beta=ACO_BETA, rho=ACO_RHO, q=ACO_Q):
    coords = np.asarray(coords, dtype=np.float64)
    n = len(coords)
    d = distance_matrix(coords)
    heuristic = 1.0 / np.maximum(d, 1e-12)
    np.fill_diagonal(heuristic, 0.0)
    pheromone = np.ones((n, n), dtype=np.float64)
    rng = np.random.default_rng(seed)
    best_tour, best_length = None, float('inf')
    for _ in range(n_iter):
        iteration_tours, iteration_lengths = [], []
        for _ant in range(n_ants):
            start = int(rng.integers(0, n))
            tour = [start]
            unvisited = np.ones(n, dtype=bool)
            unvisited[start] = False
            while unvisited.any():
                current = tour[-1]
                candidates = np.flatnonzero(unvisited)
                weights = np.power(pheromone[current, candidates], alpha) * np.power(heuristic[current, candidates], beta)
                weights = np.nan_to_num(weights, nan=0.0, posinf=0.0, neginf=0.0)
                if weights.sum() <= 0:
                    nxt = int(rng.choice(candidates))
                else:
                    nxt = int(rng.choice(candidates, p=weights / weights.sum()))
                tour.append(nxt)
                unvisited[nxt] = False
            tour = np.asarray(tour, dtype=np.int64)
            length = tour_length_np(coords, tour)
            iteration_tours.append(tour)
            iteration_lengths.append(length)
            if length < best_length:
                best_tour, best_length = tour.copy(), length
        pheromone *= (1.0 - rho)
        for tour, length in zip(iteration_tours, iteration_lengths):
            deposit = q / max(length, 1e-12)
            for k in range(n):
                a, b = int(tour[k]), int(tour[(k + 1) % n])
                pheromone[a, b] += deposit
                pheromone[b, a] += deposit
    return best_tour

# Self-tests: each route must be a permutation and all costs finite.
for _n in [20, 50, 100]:
    c = TEST_COORDS[_n][0]
    for name, t in [('NN', nearest_neighbor(c)), ('NN+2opt', two_opt(c, nearest_neighbor(c))),
                    ('ACO', ant_colony_tsp(c, seed=TEST_SEED + _n))]:
        assert sorted(np.asarray(t).tolist()) == list(range(_n)), (name, _n)
        assert np.isfinite(tour_length_np(c, t))
print('PASS: heuristic permutation and tour-length tests.')
print('ACO parameters:', {'ants': ACO_ANTS, 'iterations': ACO_ITERATIONS, 'alpha': ACO_ALPHA,
                          'beta': ACO_BETA, 'rho': ACO_RHO, 'Q': ACO_Q})

## Step 9 — Benchmark heuristics on every fixed instance

This step records one row per `(size, instance, method)`, including tour length, runtime, and the explicit ACO random seed. Raw rows are saved before any gap calculation.

In [ ]:
baseline_path = BENCH_DIR / 'baseline_per_instance.csv'
if baseline_path.exists():
    baseline_df = pd.read_csv(baseline_path)
    print('Loaded existing baseline results:', baseline_path)
else:
    baseline_rows = []
    for n, instances in TEST_COORDS.items():
        for instance_id, coords in enumerate(instances):
            seed = TEST_SEED + n * 100000 + instance_id
            for method_name, solver in [
                ('NN_start0', lambda x: nearest_neighbor(x, start=0)),
                ('NN_start0_plus_2opt', lambda x: two_opt(x, nearest_neighbor(x, start=0))),
            ]:
                t0 = time.perf_counter()
                tour = solver(coords)
                elapsed = time.perf_counter() - t0
                baseline_rows.append({'n': n, 'instance_id': instance_id, 'method': method_name,
                                      'checkpoint': '', 'decode_mode': '', 'tour_length': tour_length_np(coords, tour),
                                      'runtime_seconds': elapsed, 'seed': seed,
                                      'tour_indices': ','.join(map(str, np.asarray(tour, dtype=int).tolist()))})
            t0 = time.perf_counter()
            aco_tour = ant_colony_tsp(coords, seed=seed)
            elapsed = time.perf_counter() - t0
            baseline_rows.append({'n': n, 'instance_id': instance_id, 'method': 'ACO_fixed_parameters',
                                  'checkpoint': '', 'decode_mode': '', 'tour_length': tour_length_np(coords, aco_tour),
                                  'runtime_seconds': elapsed, 'seed': seed,
                                  'tour_indices': ','.join(map(str, np.asarray(aco_tour, dtype=int).tolist()))})
    baseline_df = pd.DataFrame(baseline_rows)
    baseline_df.to_csv(baseline_path, index=False)
    print('Saved raw heuristic results:', baseline_path)

display(baseline_df.groupby(['n', 'method']).agg(
    instances=('instance_id', 'nunique'), mean_tour_length=('tour_length', 'mean'),
    sd_tour_length=('tour_length', 'std'), mean_runtime_seconds=('runtime_seconds', 'mean')
).reset_index())

## Step 10 — Select trained checkpoints and evaluate the agent

The main benchmark evaluates the final checkpoint on all three problem sizes using:

- `greedy`: deterministic argmax decoding;
- `sampling_1`: one stochastic tour;
- `sampling_512`: 512 stochastic candidates, retaining the shortest.

For a training-progress analysis, selected intermediate checkpoints are evaluated on the 20-city test set only. If the repository's bundled external 20-city pretrained actor is present, it is evaluated separately and explicitly labeled **external reference**; it is not part of training from scratch.

In [ ]:
# Ensure Python can import modules from the cloned repository.
import os
import sys
from pathlib import Path

REPO_DIR = Path(REPO_DIR).resolve()
if not (REPO_DIR / "actor.py").is_file():
    raise FileNotFoundError(
        f"actor.py was not found in {REPO_DIR}. "
        "Rerun the repository setup cell and inspect its output."
    )
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

from types import SimpleNamespace
from tqdm.auto import tqdm
from actor import PtrNet1

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
MODEL_ARGS = {'embed': 128, 'hidden': 128, 'init_min': -0.08, 'init_max': 0.08,
              'clip_logits': 10, 'softmax_T': 1.0, 'n_glimpse': 1}
SAMPLE_BUDGET = 512


def parse_checkpoint_step(path):
    m = re.search(r'_step(\d+)_act\.pt$', Path(path).name)
    return int(m.group(1)) if m else None

def load_actor(checkpoint_path, decode_type):
    cfg = SimpleNamespace(**MODEL_ARGS, decode_type=decode_type)
    model = PtrNet1(cfg).to(DEVICE)
    try:
        state = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
    except TypeError:  # compatibility with older PyTorch
        state = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()
    return model

@torch.inference_mode()
def predict_one_instance(model, coords, decode_type, sample_count, inference_seed):
    torch.manual_seed(int(inference_seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(inference_seed))
    x0 = torch.as_tensor(coords, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    x = x0.repeat(sample_count, 1, 1)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    tours, _ = model(x, DEVICE)
    ordered = torch.gather(x, 1, tours.long().unsqueeze(-1).expand(-1, -1, 2))
    lengths = (torch.linalg.vector_norm(ordered[:, 1:] - ordered[:, :-1], dim=-1).sum(dim=1)
               + torch.linalg.vector_norm(ordered[:, 0] - ordered[:, -1], dim=-1))
    best_idx = torch.argmin(lengths)
    best_tour = tours[best_idx].detach().cpu().numpy().astype(np.int64)
    best_length = float(lengths[best_idx].item())
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    if sorted(best_tour.tolist()) != list(range(len(coords))):
        raise AssertionError('The actor produced a non-permutation tour.')
    return best_tour, best_length, elapsed

# Find snapshots created by this from-scratch run and select checkpoints for trajectory analysis.
all_full_ckpts = list((FULL_RUN_DIR / 'Pt').glob('*_step*_act.pt'))
if not all_full_ckpts:
    raise FileNotFoundError('No actor snapshots found. Finish the full training cell first.')
ckpt_by_step = {parse_checkpoint_step(p): p for p in all_full_ckpts if parse_checkpoint_step(p) is not None}
final_step = max(ckpt_by_step)
selected_steps = sorted(set(s for s in [1000, 5000, 10000, final_step] if s in ckpt_by_step))
if final_step not in selected_steps:
    selected_steps.append(final_step)
selected_steps = sorted(set(selected_steps))
print('Selected from-scratch checkpoints:', {s: str(ckpt_by_step[s]) for s in selected_steps})

external_checkpoint = REPO_DIR / 'Pt' / 'train20_1113_12_12_step14999_act.pt'
if external_checkpoint.exists():
    print('External reference checkpoint found:', external_checkpoint)
else:
    print('No bundled external checkpoint found; external comparison will be skipped.')

agent_path = BENCH_DIR / 'agent_per_instance.csv'
if agent_path.exists():
    agent_df = pd.read_csv(agent_path)
    print('Loaded existing agent results:', agent_path)
else:
    agent_rows = []
    # Selected checkpoints: trajectory analysis at n=20; final checkpoint generalizes to n=20,50,100.
    specs = []
    for step in selected_steps:
        specs.append({'path': ckpt_by_step[step], 'label': f'trained_step_{step}', 'step': step, 'sizes': [20]})
    # Ensure final checkpoint is tested on every target size.
    specs.append({'path': ckpt_by_step[final_step], 'label': f'trained_step_{final_step}', 'step': final_step, 'sizes': [20, 50, 100]})
    if external_checkpoint.exists():
        specs.append({'path': external_checkpoint, 'label': 'external_pretrained_step14999', 'step': 14999, 'sizes': [20, 50, 100]})

    seen = set()
    for spec in specs:
        for mode, samples in [('greedy', 1), ('sampling_1', 1), ('sampling_512', SAMPLE_BUDGET)]:
            # Construct a model whose city selector matches the requested inference mode.
            selector = 'greedy' if mode == 'greedy' else 'sampling'
            model = load_actor(spec['path'], selector)
            for n in spec['sizes']:
                for instance_id, coords in enumerate(TEST_COORDS[n]):
                    # Avoid duplicate final checkpoint x n=20 rows created by the trajectory list.
                    key = (spec['label'], mode, n, instance_id)
                    if key in seen:
                        continue
                    seen.add(key)
                    seed = TEST_SEED + 700000 + int(spec['step']) * 1000 + n * 100 + instance_id
                    tour, length, elapsed = predict_one_instance(model, coords, selector, samples, seed)
                    agent_rows.append({'n': n, 'instance_id': instance_id, 'method': 'DRL',
                                       'checkpoint': spec['label'], 'checkpoint_step': spec['step'],
                                       'decode_mode': mode, 'sample_count': samples,
                                       'tour_length': length, 'runtime_seconds': elapsed, 'seed': seed,
                                       'tour_indices': ','.join(map(str, tour.tolist())),
                                       'checkpoint_file': str(spec['path'])})
            del model
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()
            print('Evaluated:', spec['label'], mode, 'sizes=', spec['sizes'])
    agent_df = pd.DataFrame(agent_rows)
    agent_df.to_csv(agent_path, index=False)
    print('Saved raw agent results:', agent_path)

display(agent_df.groupby(['n', 'checkpoint', 'decode_mode']).agg(
    instances=('instance_id', 'nunique'), mean_tour_length=('tour_length', 'mean'),
    sd_tour_length=('tour_length', 'std'), mean_runtime_seconds=('runtime_seconds', 'mean')
).reset_index())

## Step 11 — Combine raw results and calculate a common per-instance gap

For each problem size and test instance, the reference is the shortest tour observed among all evaluated baseline and agent configurations for that instance. Every reported gap for that size is calculated against that same instance-level reference:

\[
 g_{i,m}=100\left(\frac{L_{i,m}}{\min_{j\in\mathcal M_n}L_{i,j}}-1\right).
\]

This is a **best-observed-reference gap**, not a gap to the exact optimum. We preserve the complete raw tour length and runtime in the outputs, so summary values can always be traced to the instance-level measurements.

In [ ]:
base_for_join = baseline_df.copy()
for col in ['checkpoint_step', 'sample_count']:
    if col not in base_for_join.columns:
        base_for_join[col] = np.nan
agent_for_join = agent_df.copy()
for col in ['checkpoint_step', 'sample_count']:
    if col not in agent_for_join.columns:
        agent_for_join[col] = np.nan

raw_df = pd.concat([base_for_join, agent_for_join], ignore_index=True, sort=False)
raw_df['tour_length'] = pd.to_numeric(raw_df['tour_length'], errors='raise')
raw_df['runtime_seconds'] = pd.to_numeric(raw_df['runtime_seconds'], errors='raise')
if raw_df.duplicated(['n', 'instance_id', 'method', 'checkpoint', 'decode_mode']).any():
    raise ValueError('Duplicate method results detected for the same test instance.')
if not np.isfinite(raw_df['tour_length']).all() or (raw_df['tour_length'] <= 0).any():
    raise ValueError('Invalid tour lengths found.')

reference = raw_df.groupby(['n', 'instance_id'])['tour_length'].min().rename('best_observed_reference')
raw_df = raw_df.join(reference, on=['n', 'instance_id'])
raw_df['gap_vs_best_observed_percent'] = 100.0 * (raw_df['tour_length'] / raw_df['best_observed_reference'] - 1.0)
raw_df['gap_vs_best_observed_percent'] = raw_df['gap_vs_best_observed_percent'].clip(lower=0.0)

raw_output = BENCH_DIR / 'all_methods_per_instance.csv'
summary_output = BENCH_DIR / 'summary_by_method.csv'
raw_df.to_csv(raw_output, index=False)
summary = (raw_df.groupby(['n', 'method', 'checkpoint', 'checkpoint_step', 'decode_mode', 'sample_count'], dropna=False)
           .agg(instances=('instance_id', 'nunique'),
                mean_tour_length=('tour_length', 'mean'),
                sd_tour_length=('tour_length', 'std'),
                mean_gap_percent=('gap_vs_best_observed_percent', 'mean'),
                sd_gap_percent=('gap_vs_best_observed_percent', 'std'),
                median_gap_percent=('gap_vs_best_observed_percent', 'median'),
                mean_runtime_seconds=('runtime_seconds', 'mean'),
                sd_runtime_seconds=('runtime_seconds', 'std'))
           .reset_index())
summary.to_csv(summary_output, index=False)

print('Raw per-instance results:', raw_output)
print('Summary results:', summary_output)
display(summary.sort_values(['n', 'mean_gap_percent']).round(4))

# Integrity checks: each size must have N_TEST_INSTANCES instances for every method/configuration.
instance_counts = raw_df.groupby(['n', 'method', 'checkpoint', 'decode_mode'], dropna=False)['instance_id'].nunique()
if not (instance_counts == N_TEST_INSTANCES).all():
    display(instance_counts[instance_counts != N_TEST_INSTANCES])
    raise AssertionError('Some method/configuration results are missing test instances.')
print('PASS: complete per-instance results for every method/configuration.')

## Step 12 — Plot benchmark results

The bar chart is a compact view of the summary, not the source of truth. The CSV files remain the authoritative record. Since the reference is based on the best observed method among all configurations available at each size, mean gaps are intended primarily for comparisons within the same problem size.

In [ ]:
# Main size-scaling comparison: final trained checkpoint, all inference modes, classical baselines.
final_label = f'trained_step_{final_step}'
plot_df = raw_df[
    raw_df['method'].isin(['NN_start0', 'NN_start0_plus_2opt', 'ACO_fixed_parameters'])
    | ((raw_df['method'] == 'DRL') & (raw_df['checkpoint'] == final_label))
].copy()
plot_df['display_method'] = plot_df.apply(
    lambda r: r['method'] if r['method'] != 'DRL' else f"DRL {r['decode_mode']}", axis=1)
plot_summary = plot_df.groupby(['n', 'display_method'])['gap_vs_best_observed_percent'].mean().reset_index()

fig, ax = plt.subplots(figsize=(13, 5.5))
for method_name, group in plot_summary.groupby('display_method'):
    group = group.sort_values('n')
    ax.plot(group['n'], group['gap_vs_best_observed_percent'], marker='o', linewidth=2, label=method_name)
ax.set_xlabel('Number of cities (n)')
ax.set_ylabel('Mean gap vs. best observed (%)')
ax.set_title('TSP benchmark — common test instances and per-instance reference')
ax.set_xticks([20, 50, 100])
ax.grid(True, alpha=.3)
ax.legend(loc='best', fontsize=8)
plt.tight_layout()
benchmark_plot = BENCH_DIR / 'benchmark_gap_summary.png'
fig.savefig(benchmark_plot, dpi=200, bbox_inches='tight')
plt.show()
print('Saved benchmark figure:', benchmark_plot)

# Training trajectory for the n=20 test set; only compare checkpoints under the same modes.
traj = summary[(summary['method'] == 'DRL') & (summary['n'] == 20) & summary['checkpoint'].str.startswith('trained_step_', na=False)].copy()
traj['checkpoint_step_num'] = pd.to_numeric(traj['checkpoint_step'], errors='coerce')
fig, ax = plt.subplots(figsize=(10, 4.5))
for mode in ['greedy', 'sampling_1', 'sampling_512']:
    g = traj[traj['decode_mode'] == mode].sort_values('checkpoint_step_num')
    if not g.empty:
        ax.plot(g['checkpoint_step_num'], g['mean_tour_length'], marker='o', label=mode)
ax.set_xlabel('Training optimizer steps')
ax.set_ylabel('Mean tour length on fixed n=20 test set')
ax.set_title('Checkpoint evaluation — fixed test instances')
ax.grid(True, alpha=.3); ax.legend()
plt.tight_layout()
trajectory_plot = BENCH_DIR / 'checkpoint_trajectory_n20.png'
fig.savefig(trajectory_plot, dpi=200, bbox_inches='tight')
plt.show()
print('Saved checkpoint trajectory:', trajectory_plot)

## Step 13 — Freeze the experiment artifacts for thesis revision

This final cell records checksums, file locations, and a short experiment summary. Copy the values and tables into the thesis only after reviewing the CSVs and confirming the configuration. Keep the raw CSVs and manifest with the thesis source archive.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

artifact_paths = [
    RUN_ROOT / 'manifest_pre.json', RUN_ROOT / 'manifest_training.json',
    RUN_ROOT / 'training_patch.diff', RUN_ROOT / 'pip-freeze.txt',
    RUN_ROOT / 'training_curves.png', raw_output, summary_output,
    BENCH_DIR / 'baseline_per_instance.csv', agent_path,
    BENCH_DIR / 'test_instances_manifest.json', benchmark_plot, trajectory_plot,
]
artifact_index = []
for p in artifact_paths:
    if p.exists():
        artifact_index.append({'path': str(p), 'bytes': p.stat().st_size, 'sha256': sha256_file(p)})
index_path = RUN_ROOT / 'artifact_index.json'
index_path.write_text(json.dumps(artifact_index, indent=2), encoding='utf-8')

print('RUN COMPLETE — artifact index:')
print(index_path)
print(json.dumps(artifact_index, indent=2))
print('\nNext: inspect the raw CSVs, verify method definitions and timings, then revise Chapter IV from these verified results.')